In [1]:
%load_ext autoreload
%autoreload 2

from tasks.diffusion import GaussianDiffusionTask
from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
from model.gaussian_diffusion import *
from evaluate import load
import einx
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import Pooling, Transformer, Normalize
from transformers import AutoModel, AutoTokenizer, T5TokenizerFast, AutoModelForCausalLM, T5Tokenizer
from datasets import load_dataset, load_from_disk
import re
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
from data.wiki import WikipediaDataset, WikipediaDatasetConfig
from torch.utils.data import DataLoader
from tasks.diffusion import GaussianDiffusionTask
from tasks.finetune import FinetuneTask
import numpy as np
from mauve import compute_mauve
from sentence_transformers import SentenceTransformer, models

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer("sentence-transformers/sentence-t5-xl")

In [10]:
model[0].auto_model.get_input_embeddings().weight.shape[1]

1024

In [11]:
import torch

In [31]:
def right_pad_dims_to(x, t):
    padding_dims = x.ndim - t.ndim
    if padding_dims <= 0:
        return t
    return t.view(*t.shape, *((1,) * padding_dims))

In [132]:
z = torch.rand(100,1024) * 100

In [135]:
alpha = 1.0

z_flat = z.view(z.shape[0], -1)
norm = z_flat.norm(dim=1, keepdim=True).detach()           
w = (norm / (z_flat.shape[1]**0.5)).clamp_min(1e-6)
z_noised = (alpha**0.5) * z + ((1 - alpha)**0.5) * right_pad_dims_to(z, w) * torch.randn_like(z)

In [136]:
z[0], z_noised[0]

(tensor([49.8031, 51.3374, 12.0739,  ..., 20.8669, 25.4563, 71.9476]),
 tensor([49.8031, 51.3374, 12.0739,  ..., 20.8669, 25.4563, 71.9476]))

In [126]:
1/17.2

0.05813953488372093

In [3]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [3]:
evals = []
for run in ["ic4jofhg", "xpyqwnw9"]:
    task = GaussianDiffusionTask.load_from_checkpoint(
                os.path.join(
                    os.environ["LATENT_CONTROL_CKPT_DIR"],
                    run, # qwen_nonorm : qrdraeyp, qwen_norm : r08t35ja, dae : 7oqrowty, dae-sem: a5i0cmio
                    "last.ckpt",
                ),
                strict=False,
            )
    task.setup()
    
    N_EVAL = 4096
    BATCH_SIZE = 64
    
    try:
        task.latent_mean = task.latent_mean.cuda()
        task.latent_scale = task.latent_scale.cuda()
    except:
        pass

    task.model = task.model.cpu()
    task.ema_model.module = task.ema_model.module.cuda()
    task.ema_model.module.eval()
    task.encoder = task.encoder.cpu().eval()
    task.decoder = task.decoder.cuda().eval()
    with torch.no_grad():
        generations = []
        for it in tqdm(range(N_EVAL // BATCH_SIZE)):
            z = sample(
                model=task.ema_model.module,
                batch_size=BATCH_SIZE,
                sampling_timesteps=150,
                sampler=task.cfg.sampler,
                schedule=task.train_schedule,
                diffusion_objective=task.cfg.diffusion_objective,
            )
            if task.cfg.normalize_latent:
                z = task.unnormalize_latent(z)
            z = z.to(torch.bfloat16)
            tokens = task.decoder.generate(
                z=z, max_length=task.cfg.max_generation_length
            )
            generations += task.decoder.tokenizer.batch_decode(
                tokens, skip_special_tokens=True
            )
    del z

    task.ema_model.module = task.ema_model.module.cpu()
    task.decoder = task.decoder.cpu()
    torch.cuda.empty_cache()

    val_seqs = [task.val_data[i] for i in range(N_EVAL)]
    mauve = compute_mauve(
        p_text=generations,
        q_text=val_seqs,
        max_text_length=150,
        batch_size=128,
        device_id=0,
        featurize_model_name='gpt2-large'
    ).mauve
    del task
    torch.cuda.empty_cache()

    evals.append((run, mauve))


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen3Model is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Flash Attention 2 only supports torch.float16 and torch.b

In [4]:
print(evals)

[('ic4jofhg', 0.8423797821584316), ('xpyqwnw9', 0.019532966738949466)]


In [4]:
task = FinetuneTask.load_from_checkpoint(
    os.path.join(
        os.environ["LATENT_CONTROL_CKPT_DIR"],
        "59vjv3km",  # qwen_nonorm : qrdraeyp, qwen_norm : r08t35ja, dae : 7oqrowty, dae-sem: a5i0cmio
        "last.ckpt",
    ),
    map_location="cuda",
    strict=False,
).to(torch.bfloat16)
task.decoder.backbone.set_attn_implementation("flash_attention_2")

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in GPT2LMHeadModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


In [8]:
# In your notebook, check for dropout layers
for name, module in task.decoder.named_modules():
    if isinstance(module, torch.nn.Dropout):
        print(f"{name}: {module}, p={module.p}, training={module.training}")

backbone.base_model.model.transformer.drop: Dropout(p=0.1, inplace=False), p=0.1, training=False
backbone.base_model.model.transformer.h.0.attn.c_attn.lora_dropout.default: Dropout(p=0.1, inplace=False), p=0.1, training=True
backbone.base_model.model.transformer.h.0.attn.c_proj.lora_dropout.default: Dropout(p=0.1, inplace=False), p=0.1, training=True
backbone.base_model.model.transformer.h.0.attn.attn_dropout: Dropout(p=0.1, inplace=False), p=0.1, training=False
backbone.base_model.model.transformer.h.0.attn.resid_dropout: Dropout(p=0.1, inplace=False), p=0.1, training=False
backbone.base_model.model.transformer.h.0.mlp.c_proj.lora_dropout.default: Dropout(p=0.1, inplace=False), p=0.1, training=True
backbone.base_model.model.transformer.h.0.mlp.dropout: Dropout(p=0.1, inplace=False), p=0.1, training=False
backbone.base_model.model.transformer.h.1.attn.c_attn.lora_dropout.default: Dropout(p=0.1, inplace=False), p=0.1, training=True
backbone.base_model.model.transformer.h.1.attn.c_proj.l

In [23]:
list(task.decoder.backbone.base_model.model.transformer.named_children())

[('wte', Embedding(50258, 1600)),
 ('wpe', Embedding(1024, 1600)),
 ('drop', Dropout(p=0.1, inplace=False)),
 ('h',
  ModuleList(
    (0-47): 48 x GPT2Block(
      (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): lora.Linear(
          (base_layer): Conv1D(nf=4800, nx=1600)
          (lora_dropout): ModuleDict(
            (default): Dropout(p=0.1, inplace=False)
          )
          (lora_A): ModuleDict(
            (default): Linear(in_features=1600, out_features=128, bias=False)
          )
          (lora_B): ModuleDict(
            (default): Linear(in_features=128, out_features=4800, bias=False)
          )
          (lora_embedding_A): ParameterDict()
          (lora_embedding_B): ParameterDict()
          (lora_magnitude_vector): ModuleDict()
        )
        (c_proj): lora.Linear(
          (base_layer): Conv1D(nf=1600, nx=1600)
          (lora_dropout): ModuleDict(
            (default): Dropout(p=0.1, inplace=Fal

In [13]:
next(task.decoder.model.children())

AttributeError: 'DecoderModel' object has no attribute 'model'

In [12]:
N_EVAL = 4096
BATCH_SIZE = 64

with torch.no_grad():
    generations = []
    for it in tqdm(range(N_EVAL // BATCH_SIZE)):
        tokens = task.decoder.generate(batch_size=BATCH_SIZE, max_length=150)
        generations += task.decoder.tokenizer.batch_decode(
            tokens, skip_special_tokens=True
        )

#val_seqs = [task.val_data[i] for i in range(N_EVAL)]
mauve = compute_mauve(
    p_text=generations,
    q_text=val_seqs,
    max_text_length=150,
    batch_size=128,
    device_id=0,
    featurize_model_name='gpt2-large'
).mauve

Featurizing q: 100%|██████████| 32/32 [00:34<00:00,  1.09s/it]
WARNING clustering 8192 points to 410 centroids: please provide at least 15990 training points


In [13]:
mauve

0.6317463420766324

In [13]:
xd = task.decoder.tokenizer.batch_encode_plus(
    val_seqs, padding="longest", return_tensors="pt"
)

In [14]:
xd['input_ids'].shape

torch.Size([4096, 146])

In [11]:
mauve

0.6614849401393732

In [10]:
val_seqs[0]

"The Treaty of Ancón was a peace treaty signed by Chile and Peru on 20 October 1883, in Ancón, near Lima. It was intended to settle the two nations' remaining territorial differences at the conclusion of their involvement in the War of the Pacific and to stabilise post-bellum relations between them."

In [9]:
generations

['On March 3, 2019, a man, identified as Joshua S. White, attempted to shoot himself with a 9mm Glock 9mm handgun while at the gym. At the time, he was "overweight, slouched in his chair and not paying attention," according to the incident report.',
 'Sakamoto, Hiroshi (2005). "Mitsuhiko Sato, the second emperor of Japan (1910–1945)". In Smith, Richard (ed.). The Cambridge History of Japan: A Comprehensive History (2nd\xa0ed.). Cambridge University Press. pp.\xa0811–814. ISBN\xa0978-0-52-043683-6.',
 "The film depicts the trials and tribulations of the family. It is based on the novel by the same name. The story takes place in an Indian boarding school and focuses on the love affair between Shanti and Prakash. The two teenagers meet each other's parents, but the parents do not welcome the two. Their families are very close-knit and love for each other is natural.",
 "The term 'Germ-free' refers to a new class of molecules with properties that are different from those of the other basic

In [5]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [6]:
task = FinetuneTask.load_from_checkpoint(
            os.path.join(
                os.environ["LATENT_CONTROL_CKPT_DIR"],
                "vb0baxg0", # qwen_nonorm : qrdraeyp, qwen_norm : r08t35ja, dae : 7oqrowty, dae-sem: a5i0cmio
                "last.ckpt",
            ),
            strict=False,
        ).to(torch.bfloat16)

FileNotFoundError: [Errno 2] No such file or directory: '/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/vb0baxg0/last.ckpt'

In [21]:
task.setup()

In [22]:
N_EVAL = 4096
BATCH_SIZE = 64

In [23]:
try:
    task.latent_mean = task.latent_mean.cuda()
    task.latent_scale = task.latent_scale.cuda()
except:
    pass

In [24]:
with torch.no_grad():
    task.ema_model.module = task.ema_model.module.cuda()
    task.ema_model.module.eval()

    generations = []
    for it in tqdm(range(N_EVAL // BATCH_SIZE)):
        z = sample(
            model=task.ema_model.module,
            batch_size=BATCH_SIZE,
            sampling_timesteps=task.cfg.sampling_timesteps,
            sampler=task.cfg.sampler,
            schedule=task.train_schedule,
            diffusion_objective=task.cfg.diffusion_objective,
        )
        if task.cfg.normalize_latent:
            z = task.unnormalize_latent(z)
        tokens = task.decoder.generate(
            z=z, max_length=task.cfg.max_generation_length
        )
        generations += task.decoder.tokenizer.batch_decode(
            tokens, skip_special_tokens=True
        )

    task.ema_model.module = task.ema_model.module.cpu()
    torch.cuda.empty_cache()

100%|██████████| 64/64 [03:31<00:00,  3.30s/it]


In [ ]:
val_seqs = [task.val_data[i] for i in range(N_EVAL)]
mauve = compute_mauve(
    p_text=generations,
    q_text=val_seqs,
    max_text_length=150,
    batch_size=128,
    device_id=0,
    featurize_model_name='gpt2-large'
).mauve

Featurizing q: 100%|██████████| 32/32 [00:34<00:00,  1.07s/it]
WARNING clustering 8192 points to 410 centroids: please provide at least 15990 training points


In [4]:
next(iter(task.decoder.parameters())).device

device(type='cuda', index=0)

In [5]:
evals = []
for run in ["ic4jofhg", "jjrzhhna", "xpyqwnw9"]:
    #["ic4jofhg", "bwjzvb81", "jjrzhhna", "xpyqwnw9"]:
    task = GaussianDiffusionTask.load_from_checkpoint(
                os.path.join(
                    os.environ["LATENT_CONTROL_CKPT_DIR"],
                    run, # qwen_nonorm : qrdraeyp, qwen_norm : r08t35ja, dae : 7oqrowty, dae-sem: a5i0cmio
                    "last.ckpt",
                ),
                strict=False,
            )
    task.setup()
    
    N_EVAL = 4096
    BATCH_SIZE = 64
    
    try:
        task.latent_mean = task.latent_mean.cuda()
        task.latent_scale = task.latent_scale.cuda()
    except:
        pass

    task.model = task.model.cpu()
    task.ema_model.module = task.ema_model.module.cuda()
    task.ema_model.module.eval()
    task.encoder = task.encoder.cpu().eval()
    task.decoder = task.decoder.cuda().eval()
    with torch.no_grad():
        generations = []
        for it in tqdm(range(N_EVAL // BATCH_SIZE)):
            z = sample(
                model=task.ema_model.module,
                batch_size=BATCH_SIZE,
                sampling_timesteps=150,
                sampler=task.cfg.sampler,
                schedule=task.train_schedule,
                diffusion_objective=task.cfg.diffusion_objective,
            )
            if task.cfg.normalize_latent:
                z = task.unnormalize_latent(z)
            z = z.to(torch.bfloat16)
            tokens = task.decoder.generate(
                z=z, max_length=task.cfg.max_generation_length
            )
            generations += task.decoder.tokenizer.batch_decode(
                tokens, skip_special_tokens=True
            )
    del z

    task.ema_model.module = task.ema_model.module.cpu()
    task.decoder = task.decoder.cpu()
    torch.cuda.empty_cache()

    val_seqs = [task.val_data[i] for i in range(N_EVAL)]
    mauve = compute_mauve(
        p_text=generations,
        q_text=val_seqs,
        max_text_length=150,
        batch_size=128,
        device_id=0,
        featurize_model_name='gpt2-large'
    ).mauve
    torch.cuda.empty_cache()

    evals.append((run, mauve))


Featurizing q: 100%|██████████| 32/32 [00:35<00:00,  1.10s/it]
WARNING clustering 8192 points to 410 centroids: please provide at least 15990 training points
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Featurizing q: 100%|██████████| 32/32 [00:35<00:00,  1.10s/it]
WARNING clustering 8192 points to 410 centroids: please provide at least 15990 training points
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
Featurizing q:  84%|████████▍ | 27/32 [00:29<00:05,  1.10s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 304.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 152.44 MiB is free. Including non-PyTorch memory, this process has 44.48 GiB memory in use. Of the allocated memory 26.41 GiB is allocated by PyTorch, and 17.58 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
evals

[('ic4jofhg', 0.8387252070350355), ('jjrzhhna', 0.010111639999093367)]

In [4]:
evals

[('bwjzvb81', 0.780873221897898)]

In [4]:
evals

[('ic4jofhg', 0.8339894941430503)]

In [ ]:
# save evals with pickle
import pickle
with open("evals.pkl", "wb") as f:
    pickle.dump(evals, f)

In [33]:
generations

['The family was "the main-contributor to several new rocket teams (including Teacup \xad—  \nwhich \nwas \nwith \n\nwitness to the \n\nfinch design\n\nComputing\n\nLectures\n\nand\n\nMonsters\n\n.\n\n\n\nIt has \n\ncollected \n\nChamberlain\n\n, \n\nJones\n\nBrowning, \n\nRichard Huckleberry, \n\nWilliam Oweley, \n\nand \n\nGladstone\n\n— \n\nTeacup\n\nand\n\nTeapot\n\nand \n\nChamber\n\nand\n\n',
 'Sansom Shayry (born February 6, 1977). book-compilation series  \nwritten about \nacoustic \nmusic, in the \nWavard \nseries, which \nwas \nwritten by \nWashner \nand \nKimber \n, \nwho \nwrote \nCarol\n.  \nThis \nwas                                                                   ',
 'PnC is accredited by a high school program to release on \xa0January\xa010,\xa02006,\xa0with a\xa0limited\xa0number\xa0of\xa0international\xa0members.\xa0PnC\xa0announced\xa0with\xa0a\xa0marriage\xa0to\xa0himself\xa0that\xa0a\xa0young\xa0woman\xa0with\xa0a\xa0hobby\xa0is\xa0at\xa0the\xa0head\xa0of\xa0time

In [ ]:
# compute likelihood of generations under electra
electra = electra.cuda()
electra.eval()
true_feats = []
for x, attn in tqdm(zip(torch.split(oracle_input['input_ids'], 16),torch.split(oracle_input['attention_mask'], 16))):
    x, attn = x.cuda(), attn.cuda()
    with torch.no_grad():
        true_feats.append(electra(x, attention_mask=attn).last_hidden_state.cpu())
true_feats = torch.cat(true_feats, dim=0)
electra = electra.cpu()
torch.cuda.empty_cache()

# compute likelihood of generations under electra
gen_inputs = tokenizer.batch_encode_plus(
    z_gen,
    padding='longest',
    return_tensors='pt',
)
electra = electra.cuda()
electra.eval()
gen_feats = []
for x, attn in tqdm(zip(torch.split(gen_inputs['input_ids'], 16),torch.split(gen_inputs['attention_mask'], 16))):
    x, attn = x.cuda(), attn.cuda()
    with torch.no_grad():
        test = electra(x, attention_mask=attn)
        gen_feats.append(test.last_hidden_state.cpu())
gen_feats = torch.cat(gen_feats, dim=0)
electra = electra.cpu()
torch.cuda.empty_cache()



In [ ]:
task.decoder = task.decoder.cuda()
task.decoder.eval()
generations = []
for i in tqdm(range(0, 8192, 64)):
    generations.append(
        task.decoder.tokenizer.batch_decode(
            task.decoder.generate(z=z[i:i+64].cuda(), max_length=task.cfg.max_generation_length),
            skip_special_tokens=True,
        )
    )
generations = [g for batch in generations for g in batch]
task.decoder = task.decoder.cpu()
torch.cuda.empty_cache()

In [ ]:
gpt2xl = AutoModelForCausalLM.from_pretrained('openai-community/gpt2-xl').cpu()
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2-xl')
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
val_seq = list(ae_task.val_data)[:1024]
oracle_input = tokenizer.batch_encode_plus(
    val_seq,
    padding='longest',
    return_tensors='pt',
)
encoder_input = ae_task.encoder.tokenizer.batch_encode_plus(
    val_seq,
    padding='longest',
    return_tensors='pt',
)
decoder_input = ae_task.decoder.tokenizer.batch_encode_plus(
    val_seq,
    padding='longest',
    return_tensors='pt',
)

In [ ]:
z = []
ae_task.encoder = ae_task.encoder.cuda()
ae_task.encoder.eval()
for x, attn in tqdm(zip(torch.split(encoder_input['input_ids'], 64), torch.split(encoder_input['attention_mask'], 64)), total=len(encoder_input['input_ids'])//64):
    x, attn = x.cuda(), attn.cuda()
    with torch.no_grad():
        feats = ae_task.encoder(input_ids=x, attention_mask=attn == 1)
    z.append(feats.cpu())
z = torch.cat(z, dim=0)
ae_task.encoder = ae_task.encoder.cpu()
torch.cuda.empty_cache()

100%|██████████| 16/16 [00:14<00:00,  1.12it/s]


In [ ]:
ae_task.decoder = ae_task.decoder.cuda()
ae_task.decoder.eval()
generations = []
for i in tqdm(range(0, len(z), 64)):
    generations.append(
        ae_task.decoder.tokenizer.batch_decode(
            ae_task.decoder.generate(z=z[i:i+64].cuda(), max_length=ae_task.cfg.max_generation_length),
            skip_special_tokens=True,
        )
    )
generations = [g for batch in generations for g in batch]
ae_task.decoder = ae_task.decoder.cpu()
torch.cuda.empty_cache()

100%|██████████| 16/16 [00:50<00:00,  3.14s/it]


In [ ]:
# compute likelihood of generations under gpt2-xl
gpt2xl = gpt2xl.cuda()
gpt2xl.eval()
log_likelihoods = []
for x, attn in tqdm(zip(torch.split(oracle_input['input_ids'], 16),torch.split(oracle_input['attention_mask'], 16))):
    x, attn = x.cuda(), attn.cuda()
    with torch.no_grad():
        outputs = gpt2xl(input_ids=x, attention_mask=attn, labels=x)
        log_likelihoods.append(outputs.loss.cpu())
log_likelihoods = torch.stack(log_likelihoods).mean()
gpt2xl = gpt2xl.cpu()
torch.cuda.empty_cache()

64it [00:25,  2.51it/s]


In [ ]:
log_likelihoods

tensor(8.0722)

In [ ]:
true_features = []
for x, attn in tqdm(zip(torch.split(val_tokens['input_ids'], 256),torch.split(val_tokens['attention_mask'], 256))):
    x, attn = x.cuda(), attn.cuda()
    with torch.no_grad():
        feats = electra(input_ids=x, attention_mask=attn).last_hidden_state
    true_features.append(feats.cpu())

40it [00:51,  1.28s/it]


In [ ]:
true_features = torch.cat(true_features, dim=0)

In [ ]:
mauve = compute_mauve(
    p_text=generations,
    q_features=self.val_feats,
    max_text_length=self.cfg.max_generation_length,
    batch_size=128,
    device_id=0,
    featurize_model_name="gpt2-large",
).mauve
self.decoder = self.decoder.cpu()

In [ ]:
z = self.encoder(batch["input_ids_enc"], batch["attention_mask_enc"])

1024

In [ ]:
dataset = WikipediaDataset(max_length=128)

In [ ]:
enc_tokenizer = T5Tokenizer.from_pretrained("thesephist/contra-bottleneck-t5-xl-wikipedia")
dec_tokenizer = AutoTokenizer.from_pretrained("gpt2-large")
dec_tokenizer.pad_token = dec_tokenizer.eos_token

In [ ]:
dl = DataLoader(
    dataset,
    batch_size=128,
    num_workers=4,
    shuffle=True,
    collate_fn=dataset.get_collate_and_tokenize_fn(
        enc_tokenizer=enc_tokenizer,
        dec_tokenizer=dec_tokenizer,
    ),
)

In [ ]:
dl_iter = iter(dl)

In [ ]:
batch = next(dl_iter)

In [ ]:
batch

{'input_ids_enc': tensor([[ 2474,  1496,     6,  ...,     0,     0,     0],
         [11340,   138,   644,  ...,     0,     0,     0],
         [   37,     3,  4365,  ...,     0,     0,     0],
         ...,
         [ 5104,  7242,  1896,  ...,     0,     0,     0],
         [ 2659,    49,    29,  ...,     0,     0,     0],
         [20650,  1783,    47,  ...,     0,     0,     0]]),
 'attention_mask_enc': tensor([[ True,  True,  True,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False],
         ...,
         [ True,  True,  True,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False],
         [ True,  True,  True,  ..., False, False, False]]),
 'input_ids_dec': tensor([[50256, 15262,  2516,  ..., 50256, 50256, 50256],
         [50256, 30380,   282,  ..., 50256, 50256, 50256],
         [50256,   464,   936,  ..., 50256, 50256, 50256],
         ...,
         [502

In [ ]:
ds = load_from_disk("/network/scratch/l/leo.gagnon/sentence_diffusion/data/wikipedia-paragraphs-filtered")

In [ ]:
ds['train']

Dataset({
    features: ['input_ids'],
    num_rows: 17631130
})

In [ ]:
ds['train'][421121]['input_ids']

'Year 489 BC was a year of the pre-Julian Roman calendar. At the time, it was known as the Year of the Consulship of Iullus and Rufus (or, less frequently, year 265 Ab urbe condita). The denomination 489 BC for this year has been used since the early medieval period, when the Anno Domini calendar era became the prevalent method in Europe for naming years.'

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def last_token_pool(last_hidden_states: torch.Tensor,
                 attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-8B', padding_side='left')
model = AutoModel.from_pretrained('Qwen/Qwen3-Embedding-8B')

Fetching 4 files:   0%|          | 0/4 [01:13<?, ?it/s]


In [ ]:
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

In [ ]:
task = ""

documents = [
    get_detailed_instruct()
]

In [ ]:
enc_tokenizer = T5Tokenizer.from_pretrained("thesephist/contra-bottleneck-t5-xl-wikipedia")

ds = load_dataset("singletongue/wikipedia-paragraphs", 'enwiki-20250901')

In [ ]:
ds['train'] = ds['train'].take(10000)

In [ ]:
def extract_and_filter_paragraphs_batched(examples):
    """
    Extract individual paragraphs from documents and filter them.
    Returns a flattened structure where each item is a single paragraph.
    """
    filtered_paragraphs = []
    flat_paragraphs = [item for sublist in examples['paragraph_texts'] for item in sublist]
    enc_tokens = enc_tokenizer.batch_encode_plus(flat_paragraphs)['input_ids']
        
    for i, text in enumerate(flat_paragraphs):
        if not text[0].isupper():
            continue
        
        # Check encoder tokenization (T5)
        if len(enc_tokens[i]) < 50 or len(enc_tokens[i]) > 100:
            continue

        filtered_paragraphs.append(text)

    return {"text": filtered_paragraphs}


# Save the paragraph dataset
#paragraph_ds.save_to_disk("/network/scratch/l/leo.gagnon/sentence_diffusion/data/wikipedia-paragraphs-individual-filtered")

In [ ]:
examples = ds['train'].take(10)
flat_paragraphs = [item for sublist in examples['paragraph_texts'] for item in sublist]

In [ ]:
flat_paragraphs += [[]]

In [ ]:
enc_tokenizer.batch_encode_plus([])['input_ids']

ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided []

In [ ]:
[item for sublist in [] for item in sublist] == []

True

In [ ]:

# Apply the transformation to create a flattened paragraph dataset
print("Extracting and filtering individual paragraphs...")
paragraph_ds = ds.map(
    extract_and_filter_paragraphs_batched,
    batched=True,
    batch_size=100,
    num_proc=1,
    remove_columns=ds['train'].column_names,  # Remove original columns
    desc="Extracting individual paragraphs"
)

# The result will have many more items since we're flattening
print(f"Total filtered paragraphs: {len(paragraph_ds['train'])}")

Extracting and filtering individual paragraphs...


Extracting individual paragraphs (num_proc=4):   0%|          | 0/10000 [00:13<?, ? examples/s]


KeyboardInterrupt: 

KeyboardInterrupt: 

In [ ]:
paragraph_ds['train'][3822]['text']

"August 30\nConstitution Day (Kazakhstan)\nConstitution Day  (Turks and Caicos Islands)\nIndependence Day (Tatarstan, Russia, unrecognized)\nInternational Day of the Disappeared (International)\nPopular Consultation Day (East Timor)\nSaint Rose of Lima's Day (Peru)\nVictory Day (Turkey)"

In [ ]:
def process(examples):
    # Process a batch of examples
    processed_texts = []
    for text in examples['text']:
        # Your processing logic
        processed_text = text.strip().lower()
        processed_texts.append(processed_text)
    
    return {"processed_text": processed_texts}

# Apply processing iteratively in batches
processed_ds = ds.map(
    process,
    num_proc=4  # Use multiprocessing
)

In [ ]:
data = []
for item in tqdm(ds['train']['paragraph_texts'], total=len(ds['train'])):
    tok_p = enc_tokenizer.batch_encode_plus(item)['input_ids']
    for i in range(len(tok_p)):
        p = tok_p[i]
        if (len(p) > 64) and (len(p) < 128) and (item[i][0].isupper()):
            data.append(item[i])

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f96c29e7790>>
Traceback (most recent call last):
  File "/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
ds['train'][:2]['paragraph_texts']

[['Anarchism is a political philosophy and movement that seeks to abolish all institutions that perpetuate authority, coercion, or hierarchy, primarily targeting the state and capitalism. Anarchism advocates for the replacement of the state with stateless societies and voluntary free associations. A historically left-wing movement, anarchism is usually described as the libertarian wing of the socialist movement (libertarian socialism).',
  "Although traces of anarchist ideas are found all throughout history, modern anarchism emerged from the Enlightenment. During the latter half of the 19th and the first decades of the 20th century, the anarchist movement flourished in most parts of the world and had a significant role in workers' struggles for emancipation. Various anarchist schools of thought formed during this period. Anarchists have taken part in several revolutions, most notably in the Paris Commune, the Russian Civil War and the Spanish Civil War, whose conclusion marked the end 

In [ ]:
seq = [p for p in ds['train'][1]['paragraph_texts'] if (len(p) > 100) and (len(p) < 500) and (p[0].isupper())]
xd = dec_tokenizer.batch_encode_plus(seq)

In [ ]:
def filter_by_token_length_batched(examples):
    """Filter examples to keep only those with GPT2 tokenization < 512 tokens"""
    # Clean the texts by normalizing whitespace
    cleaned_texts = [text.strip() for text in examples['text']]
    
    # Tokenize the batch
    tokenized = tokenizer(
        cleaned_texts, 
        truncation=False,  # Don't truncate so we can get actual lengths
        padding=False,     # Don't pad for length calculation
        return_tensors=None  # Return lists for easier processing
    )

    # Check which examples have less than 256 tokens
    keep_mask = [len(tokens) < 256 for tokens in tokenized['input_ids']]
    
    return keep_mask

# Apply the filter in batches for efficiency with multiprocessing
print("Filtering dataset...")
filtered_ds = ds.filter(
    filter_by_token_length_batched, 
    batched=True, 
    batch_size=1000,
    num_proc=4,  # Use 8 processes - adjust based on your CPU cores
    desc="Filtering by token length"
)

print(f"Original dataset size: {len(ds)}")
print(f"Filtered dataset size: {len(filtered_ds)}")
print(f"Kept {len(filtered_ds)/len(ds)*100:.1f}% of examples")

In [ ]:
[len(x) for x in xd['input_ids']]

[72,
 113,
 91,
 82,
 37,
 30,
 67,
 94,
 91,
 95,
 93,
 81,
 36,
 77,
 110,
 65,
 24,
 84,
 50,
 89,
 63,
 70,
 62,
 25,
 85,
 69,
 83,
 74,
 65,
 106,
 28,
 120,
 67,
 65,
 49,
 74,
 42,
 40,
 84]

In [ ]:
seq

['While directional-hemispherical reflectance factor is calculated for a single angle of incidence (i.e., for a given position of the Sun), albedo is the directional integration of reflectance over all solar angles in a given period. The temporal resolution may range from seconds (as obtained from flux measurements) to daily, monthly, or annual averages.',
 'Unless given for a specific wavelength (spectral albedo), albedo refers to the entire spectrum of solar radiation. Due to measurement constraints, it is often given for the spectrum in which most solar energy reaches the surface (between 0.3 and 3 μm). This spectrum includes visible light (0.4–0.7 μm), which explains why surfaces with a low albedo appear dark (e.g., trees absorb most radiation), whereas surfaces with a high albedo appear bright (e.g., snow reflects most radiation).',
 'Ice–albedo feedback is a positive feedback climate process where a change in the area of ice caps, glaciers, and sea ice alters the albedo and surfa

In [ ]:
from datasets import load_dataset
from transformers import GPT2Tokenizer
import re

# Load the dataset and tokenizer
ds = load_dataset("agentlans/wikipedia-paragraphs-complete", 'all', split='train')
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Add pad token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def filter_by_token_length_batched(examples):
    """Filter examples to keep only those with GPT2 tokenization < 512 tokens"""
    # Clean the texts by normalizing whitespace
    cleaned_texts = [text.strip() for text in examples['text']]
    
    # Tokenize the batch
    tokenized = tokenizer(
        cleaned_texts, 
        truncation=False,  # Don't truncate so we can get actual lengths
        padding=False,     # Don't pad for length calculation
        return_tensors=None  # Return lists for easier processing
    )

    # Check which examples have less than 256 tokens
    keep_mask = [len(tokens) < 256 for tokens in tokenized['input_ids']]
    
    return keep_mask

# Apply the filter in batches for efficiency with multiprocessing
print("Filtering dataset...")
filtered_ds = ds.filter(
    filter_by_token_length_batched, 
    batched=True, 
    batch_size=1000,
    num_proc=4,  # Use 8 processes - adjust based on your CPU cores
    desc="Filtering by token length"
)

print(f"Original dataset size: {len(ds)}")
print(f"Filtered dataset size: {len(filtered_ds)}")
print(f"Kept {len(filtered_ds)/len(ds)*100:.1f}% of examples")

Filtering dataset...


Filtering by token length (num_proc=4):   0%|          | 0/1429479 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1060 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1646 > 1024). Running this sequence through the model will result in indexing errors
Filtering by token length (num_proc=4):   0%|          | 1000/1429479 [00:02<1:00:16, 395.01 examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1071 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1162 > 1024). Running this sequence through the model will result in indexing errors
Filtering by token length (num_proc=4): 100%|██████████| 1429479/14294

Original dataset size: 1429479
Filtered dataset size: 750652
Kept 52.5% of examples


In [ ]:
filtered_ds.save_to_disk("/network/scratch/l/leo.gagnon/sentence_diffusion/data/wikipedia-paragraphs-complete-filtered-256")

Saving the dataset (2/2 shards): 100%|██████████| 750652/750652 [00:03<00:00, 233648.14 examples/s]


In [ ]:
# load dataset
from datasets.load import load_from_disk


ds = load_from_disk("/network/scratch/l/leo.gagnon/sentence_diffusion/data/wikipedia-paragraphs-complete-filtered-512")

In [ ]:
# Optional: Save the filtered dataset to disk
# filtered_ds.save_to_disk("wiki_paragraphs_filtered_512")

# Optional: Check some examples and their token lengths
print("\nSample of filtered examples with token counts:")
for i in range(min(5, len(filtered_ds))):
    text = filtered_ds[i]['text']
    tokens = tokenizer.encode(text)
    print(f"Example {i}: {len(tokens)} tokens")
    print(f"Text preview: {text[:100]}...")
    print("-" * 50)

In [ ]:
class WikipediaDataset(Dataset):
    def __init__(self, max_length=512):
        self.dataset = load_dataset(
            "agentlans/wikipedia-paragraphs-complete", "all", split="train"
        )
        self.max_length = max_length

    def get_collate_and_tokenize_fn(self, enc_tokenizer, dec_tokenizer):
        def collate_fn(texts):
            batch_enc = enc_tokenizer.batch_encode_plus(
                texts,
                truncation=True,
                padding=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            out = {
                "input_ids_enc": batch_enc["input_ids"],
                "attention_mask_enc": batch_enc["attention_mask"],
            }
            batch_dec = dec_tokenizer.batch_encode_plus(texts)
            input_ids_dec = pad_sequence(
                [
                    torch.LongTensor(
                        [dec_tokenizer.bos_token_id]
                        + ids
                        + [dec_tokenizer.eos_token_id]
                    )
                    for ids in batch_dec["input_ids"]
                ],
                padding_value=dec_tokenizer.pad_token_id,
                batch_first=True,
            )
            attention_mask_dec = (input_ids_dec != dec_tokenizer.pad_token_id).long()
            out["input_ids_dec"] = input_ids_dec
            out["attention_mask_dec"] = attention_mask_dec

            return out

        return collate_fn

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]["text"]

In [ ]:
wiki = WikipediaDataset(512)

In [ ]:
if dec_tokenizer.pad_token_id is None:
    dec_tokenizer.add_special_tokens({"pad_token": "[PAD]"})

In [ ]:
dl = DataLoader(wiki, batch_size=32, shuffle=True, num_workers=1, collate_fn=wiki.get_collate_and_tokenize_fn(enc_tokenizer, dec_tokenizer))

In [ ]:
iter_dl = iter(dl)

In [ ]:
batch = next(iter_dl)

In [ ]:
tokd

Dataset({
    features: ['title', 'input_ids', 'attention_mask'],
    num_rows: 10000
})

In [ ]:
tokd = tokd.remove_columns(['title', 'attention_mask'])

In [ ]:
tokd.save_to_disk('wiki-tokd-10k')

Saving the dataset (1/1 shards): 100%|██████████| 10000/10000 [00:00<00:00, 227297.82 examples/s]


In [ ]:
from datasets.load import load_from_disk


ds = load_from_disk("wiki-tokd-10k")

In [ ]:
def collate(features):
    ids = [torch.tensor(f["input_ids"], dtype=torch.long) for f in features]
    padded = torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attn = (padded != tokenizer.pad_token_id).long()

    return {
        "input_ids": padded,           # CPU
        "attention_mask": attn,        # CPU
    }

loader = DataLoader(
    ds,
    batch_size=64,            # tune
    shuffle=False,            # shuffle at dataset level
    num_workers=1,            # 4–8 per GPU is typical
    pin_memory=True,          # enables non_blocking H2D copies
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate
)

In [ ]:
dl_iter = iter(loader)

In [ ]:
len(ds['train'])

1429479

In [ ]:
ds

In [ ]:
class WikiParagraphsDataset(torch.utils.data.Dataset):
    def __init__(self, max_length=512):
        
        self.max_length = max_length

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        text = item['text']
        text = re.sub(r'\s+', ' ', text).strip()
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask
        }

In [ ]:
len(tokenizer.encode(ds['train'][10000]['text']))

235

In [ ]:
ds['train'][312321]['text']

'The World Health Organization, in conjunction with the Food and Agriculture Organization, published guidelines that can be effectively represented in a food pyramid relating to objectives in order to prevent obesity, improper nutrition, chronic diseases and dental caries based on meta-analysis though they represent it as a table rather than as a "pyramid". The structure is similar in some respects to the USDA food pyramid, but there are clear distinctions between types of fats, and a more dramatic distinction where carbohydrates are categorized on the basis of free sugars versus sugars in their natural form. Some food substances are singled out due to the impact on the target issues that the "pyramid" is meant to address. In a later revision, however, some recommendations are omitted as they automatically follow other recommendations while other sub-categories are added. The reports quoted here explain that where there is no stated lower limit in the table below, there is no requireme

In [ ]:
max = 0
id = None
for i in range(10000):
    l = len(filtered_ds['train'][i]['abstract'])
    if l > max:
        max = l
        id = i
print(max)

1875


Filter: 100%|██████████| 1999486/1999486 [00:29<00:00, 67089.35 examples/s]


In [ ]:
len(ds['train'])

1999486

In [ ]:
def filter_by_token_length_batched(examples):
    # Clean all texts in the batch
    cleaned_texts = [re.sub(r'\s+', ' ', abstract).strip() for abstract in examples['abstract']]
    
    # Tokenize the entire batch at once
    tokens = tok(cleaned_texts, return_tensors="pt", padding=True, truncation=False)
    
    # Check token lengths for each example in the batch
    token_lengths = (tokens['attention_mask'].sum(dim=1) < 300).tolist()
    
    return token_lengths

# Apply batched filtering
filtered_ds = ds.filter(filter_by_token_length_batched, batched=True, batch_size=1000)

Filter: 100%|██████████| 1999486/1999486 [26:35<00:00, 1253.33 examples/s]


In [ ]:
filtered_ds.save_to_disk("/network/scratch/l/leo.gagnon/sentence_diffusion/data/filtered_arxiv.parquet")

Saving the dataset (4/4 shards): 100%|██████████| 1577821/1577821 [00:22<00:00, 70274.22 examples/s]


In [ ]:
filtered_ds = ds.filter(lambda example: len(example['abstract']) <= 5000)

'  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from the Fermilab Tevatron, and predictions are made for\nmore detailed tests with CDF and DO data. Predictions are shown for\ndistributions of diphoton pairs produced at the energy of the Large Hadron\nCollider (LHC). Distributions of the diphoton pairs from the decay of a Higgs\nboson are contrasted with those produced from QCD processes at the LHC, showing\nthat enhanced sensitivity to the signal can be obtained with judicious\nselection of events.\n'

In [ ]:
def clean_abstract(example):
    example["abstract"] = re.sub(r"\n", " ", example["abstract"]).strip()
    return example

filtered_ds = filtered_ds.map(clean_abstract)

Map: 100%|██████████| 1577821/1577821 [03:39<00:00, 7198.13 examples/s]


In [ ]:
filtered_ds['train'][560077]['abstract']

'Matrix integrals used in random matrix theory for the study of eigenvalues of Hermitian ensembles have been shown to provide $\\tau$-functions for several hierarchies of integrable equations. In this article, we extend this relation by showing that such integrals can also provide $\\tau$-functions for the discrete KP hierarchy and a coupled version of the same hierarchy obtained through the process of Pfaffianization. To do so, we consider the first equation of the discrete KP hierarchy, the Hirota-Miwa equation. We write the Wronskian determinant solutions to the Hirota-Miwa equation and consider a particular form of matrix integrals, which we show is an example of those Wronskian solutions. The argument is then generalized to the whole hierarchy. A similar strategy is used for the Pfaffianized version of the hierarchy except that in that case, the solutions are written in terms of Pfaffians rather than determinants.'

In [ ]:
tok = T5TokenizerFast.from_pretrained('sentence-transformers/sentence-t5-large')

In [ ]:
txt = re.sub(r'\n', ' ', ds['train'][id]['abstract']).strip()

In [ ]:
len(tok.

587

In [ ]:
import torch
from tasks.autoencoder import AETask
import os
import einops

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/",
                '02cmdbct/',
                "last.ckpt",
            ),
            strict=False,
        )

Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


InstantiationException: Error in call to target 'model.encoder.DAEEncoder':
ConfigAttributeError('Missing key k\n    full_key: k\n    object_type=dict')

In [ ]:
ae_task.eval()
ae_task.setup()

In [ ]:
batch = next(iter(ae_task.val_dataloader()))

1.0

In [ ]:
z

tensor([[-0.0529,  0.0096, -0.0145,  ..., -0.0217, -0.0202,  0.0476],
        [-0.0393, -0.0094, -0.0034,  ..., -0.0420, -0.0053,  0.0393],
        [-0.0289,  0.0300, -0.0142,  ..., -0.0259,  0.0065,  0.0408],
        ...,
        [-0.0558, -0.0130,  0.0035,  ..., -0.0381,  0.0059,  0.0214],
        [-0.0032, -0.0062, -0.0196,  ..., -0.0406, -0.0018,  0.0415],
        [-0.0403,  0.0044, -0.0037,  ..., -0.0626, -0.0095,  0.0276]],
       device='cuda:0')

In [ ]:
z = ae_task.encoder(batch["input_ids_enc"].cuda(), batch["attention_mask_enc"].cuda())

In [ ]:
# Compute cache for z
prompt = ae_task.decoder.prompt_generator(z)
cache = ae_task.decoder.backbone(
    inputs_embeds=prompt,
    use_cache=True,
).past_key_values
# Causal attention mask for the prompt + BOS
attention_mask = torch.tril(torch.ones((64, 17), device=z.device))


In [ ]:
# Autoregressive generation from BOS token with cached z (nucleus sampling)
bos = torch.full(
    (z.shape[0], 1),
    ae_task.decoder.tokenizer.bos_token_id,
    device=z.device,
    dtype=torch.long,
)
output = ae_task.decoder.backbone.generate(
    input_ids=bos,
    past_key_values=cache,
    attention_mask=attention_mask,
    cache_position=torch.tensor([prompt.shape[1]], device=z.device),
    max_length=150,
    do_sample=True,
    top_p=0.92,
    top_k=50,
    num_beams=1,
    temperature=0.9,
    return_dict_in_generate=True,
    use_cache=True,
    pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
    eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
)

output = output.sequences[:, 1:]  # Remove BOS token
xd = ae_task.decoder.tokenizer.batch_decode(output, skip_special_tokens=True)

OutOfMemoryError: CUDA out of memory. Tried to allocate 28.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 26.44 MiB is free. Including non-PyTorch memory, this process has 44.61 GiB memory in use. Of the allocated memory 42.37 GiB is allocated by PyTorch, and 1.74 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
xd

[' Pres Pres Pres Pres Pres Pres P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P P',
 ' Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol Vol P Vol Vol Vol Vol Vol Vol\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n',
 ' S S S S S S S S S S S S S S S S S S S S S S S S S S S S S\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n

In [ ]:
from functools import partial
from model.gaussian_diffusion import time_to_alpha, get_sampling_schedule
import torch

In [ ]:
time_to_alpha(t=torch.tensor(0.5), alpha_schedule=get_sampling_schedule("cosine"),scale=3.0)

tensor(0.9000)

In [ ]:
from transformers import T5EncoderModel, T5Tokenizer, T5Model, AutoModelForCausalLM
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sentence_transformers.models import InputModule

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class PretrainedDAE(InputModule):
    def __init__(self, name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia",
            trust_remote_code=True,
        )
        self.tokenizer = T5Tokenizer.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia"
        )
        del self.model.decoder, self.model.dec_emb, self.model.lm_head

    def tokenize(self, texts):
        return self.tokenizer.batch_encode_plus(
            texts, return_tensors="pt", padding=True, 
        )

    def save(self, path):
        pass

    def get_sentence_embedding_dimension(self):
        return self.model.bottleneck.out_proj.out_features

    def forward(self, features):
        hidden_states = self.model.encoder(**features).last_hidden_state
        attention_mask = features["attention_mask"]

        hidden_states = hidden_states.repeat(
            attention_mask.shape[0] // hidden_states.shape[0], 1, 1
        )  # during contrastive search, attn mask can have higher batch size than hidden_state
        mask_expanded = attention_mask.to(dtype=hidden_states.dtype).unsqueeze(-1).expand(hidden_states.shape)
        mean_pooled_embedding = torch.sum(
            hidden_states * mask_expanded, 1
        ) / torch.clamp(mask_expanded.sum(1), min=1e-9)
        unscaled_latent, attn_weights = self.model.bottleneck(
            mean_pooled_embedding.unsqueeze(1),
            hidden_states,
            hidden_states,
            need_weights=False,
            # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
            attn_mask=attention_mask.to(dtype=hidden_states.dtype)
            .unsqueeze(1)
            .repeat_interleave(self.model.num_heads, dim=0),
        )
        latent = self.model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

        return {"sentence_embedding": latent.squeeze(1)}

In [ ]:
semb = PretrainedDAE("xl")
test = SentenceTransformer(modules=[semb], model_kwargs={"torch_dtype": "float16"})

Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.35it/s]


In [ ]:
a = test.encode(sentences=["Allo mon nom est Léo", "Wassup my boii"], convert_to_tensor=True)

In [ ]:
b = test.encode(sentences=["Allo mon nom est Léo", "Wassup my boii"], convert_to_tensor=True)

In [ ]:
(a - b.to(dtype=torch.float32)).norm(dim=1)

tensor([1.4720, 1.3557], device='cuda:0')

In [ ]:
a.norm(dim=1)

tensor([1.7872, 1.7872], device='cuda:0')

In [ ]:
b.to(dtype=torch.float32).norm(dim=1)

tensor([1.7869, 1.7870], device='cuda:0')

In [ ]:
from sentence_transformers import SentenceTransformer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
tok = T5Tokenizer.from_pretrained("thesephist/contra-bottleneck-t5-xl-wikipedia")
model = AutoModelForCausalLM.from_pretrained(
    f"thesephist/contra-bottleneck-t5-xl-wikipedia", trust_remote_code=True
).cuda()

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.22it/s]


In [ ]:
inputs = tok.batch_encode_plus(["Allo mon nom est Léo", "Wassup my boii"], return_tensors='pt', padding=True).to('cuda')
decoder_inputs = tok.batch_encode_plus(['']*2, return_tensors='pt', padding=True)
b=model(
    **inputs,
    decoder_input_ids=decoder_inputs['input_ids'],
    encode_only=True,
)

In [ ]:
a -b

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0',
       grad_fn=<SubBackward0>)

In [ ]:
del model.decoder

In [ ]:
encoder_outputs = model.encoder(**inputs)

In [ ]:
hidden_states = encoder_outputs.last_hidden_state
attention_mask = inputs['attention_mask']

In [ ]:
from sentence_transformers.models import Transformer

In [ ]:
hidden_states = hidden_states.repeat(
    attention_mask.shape[0] // hidden_states.shape[0], 1, 1
)  # during contrastive search, attn mask can have higher batch size than hidden_state
mask_expanded = attention_mask.float().unsqueeze(-1).expand(hidden_states.shape)
mean_pooled_embedding = torch.sum(hidden_states * mask_expanded, 1) / torch.clamp(
    mask_expanded.sum(1), min=1e-9
)
unscaled_latent, attn_weights = model.bottleneck(
    mean_pooled_embedding.unsqueeze(1),
    hidden_states,
    hidden_states,
    need_weights=False,
    # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
    attn_mask=attention_mask.float()
    .unsqueeze(1)
    .repeat_interleave(model.num_heads, dim=0),
)
latent = model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

In [ ]:
class PretrainedDAE(InputModule):
    def __init__(self, name):
        super().__init__()
        self.model = AutoModelForCausalLM.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia", trust_remote_code=True
        )
        self.tokenizer = T5Tokenizer.from_pretrained(
            f"thesephist/contra-bottleneck-t5-{name}-wikipedia"
        )
        del self.model.decoder, self.model.dec_emb, self.model.lm_head

    def tokenize(self, texts):
        return self.tokenizer.batch_encode_plus(texts, return_tensors="pt", padding=True)
    
    def save(self, path):
        pass
    
    def forward(self, features):
        hidden_states = self.model.encoder(**features).last_hidden_state
        attention_mask = features["attention_mask"]

        hidden_states = hidden_states.repeat(
            attention_mask.shape[0] // hidden_states.shape[0], 1, 1
        )  # during contrastive search, attn mask can have higher batch size than hidden_state
        mask_expanded = attention_mask.float().unsqueeze(-1).expand(hidden_states.shape)
        mean_pooled_embedding = torch.sum(hidden_states * mask_expanded, 1) / torch.clamp(
            mask_expanded.sum(1), min=1e-9
        )
        unscaled_latent, attn_weights = self.model.bottleneck(
            mean_pooled_embedding.unsqueeze(1),
            hidden_states,
            hidden_states,
            need_weights=False,
            # torch MHA attn_mask has opposite signs to HF T5 masks... sigh
            attn_mask=attention_mask.float()
            .unsqueeze(1)
            .repeat_interleave(self.model.num_heads, dim=0),
        )
        latent = self.model.bottleneck_scale * F.normalize(unscaled_latent, p=2, dim=2)

        return latent.squeeze(1)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.65it/s]


/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/sentence_transformers/SentenceTransformer.py:1080: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  embeddings = out_features[output_value]


IndexError: too many indices for tensor of dimension 2

In [ ]:
from sentence_transformers import SentenceTransformer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/sentence-t5-base', model_kwargs={"torch_dtype": "float16"})

In [ ]:
batch = list(model.children())[0].tokenizer(sentences, return_tensors="pt", padding=True)

In [ ]:
modules = list(model.children())

In [ ]:
model.tokenizer

T5TokenizerFast(name_or_path='sentence-transformers/sentence-t5-base', vocab_size=32100, model_max_length=256, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<extra_id_0>', '<extra_id_1>', '<extra_id_2>', '<extra_id_3>', '<extra_id_4>', '<extra_id_5>', '<extra_id_6>', '<extra_id_7>', '<extra_id_8>', '<extra_id_9>', '<extra_id_10>', '<extra_id_11>', '<extra_id_12>', '<extra_id_13>', '<extra_id_14>', '<extra_id_15>', '<extra_id_16>', '<extra_id_17>', '<extra_id_18>', '<extra_id_19>', '<extra_id_20>', '<extra_id_21>', '<extra_id_22>', '<extra_id_23>', '<extra_id_24>', '<extra_id_25>', '<extra_id_26>', '<extra_id_27>', '<extra_id_28>', '<extra_id_29>', '<extra_id_30>', '<extra_id_31>', '<extra_id_32>', '<extra_id_33>', '<extra_id_34>', '<extra_id_35>', '<extra_id_36>', '<extra_id_37>', '<extra_id_38>', '<extra_id_39>', '<extra_id_40>', '<extra_id_41>', '<extra_id_42

In [ ]:
out.sentence_embedding

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16, grad_fn=<DivBackward0>)

In [ ]:
xd =list(model.children())[0].forward(
    batch.to('cuda')
)

In [ ]:
list(model.children())[0].encode(sentences, convert_to_tensor=True)

AttributeError: 'Transformer' object has no attribute 'encode'

In [ ]:
from torch.nn import Sequential

In [ ]:
seq = Sequential(*list(model.children())[1:])

In [ ]:
out = seq(xd)

In [ ]:
out_ = model.encode(sentences, convert_to_tensor=True)

In [ ]:
out_

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16)

In [ ]:
out.sentence_embedding

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16, grad_fn=<DivBackward0>)

In [ ]:
res = xd.token_embeddings

In [ ]:
embeddings = model.encode(sentences, convert_to_tensor=True)

tensor([[-0.0091,  0.0191,  0.0266,  ..., -0.0087, -0.0560, -0.0216],
        [-0.0078,  0.0302,  0.0313,  ..., -0.0124, -0.0624, -0.0059]],
       device='cuda:0', dtype=torch.float16)

In [ ]:
from transformers import T5EncoderModel, T5Tokenizer, T5Model
from sentence_transformers import SentenceTransformer
# import AutoModel
from transformers import AutoModel, AutoTokenizer

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = SentenceTransformer(
    "sentence-transformers/sentence-t5-xl",
    device="cuda",
    model_kwargs={"torch_dtype": "float16"},
)

In [ ]:
model.get_sentence_embedding_dimension()

768

In [ ]:
list(model.children())

[Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'T5EncoderModel'}),
 Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True}),
 Dense({'in_features': 1024, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity'}),
 Normalize()]

In [ ]:
input_ids= tok("Hello, my dog is cute", return_tensors="pt").input_ids
xd = model(input_ids)

In [ ]:
xd.last_hidden_state.shape

torch.Size([1, 7, 1024])

In [ ]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [ ]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/cm9ujm08/"
                "last.ckpt",
            ),
            strict=False
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


In [ ]:
ae_task = ae_task.cuda()

In [ ]:
ae_task.setup()

In [ ]:
batch = next(iter(ae_task.val_dataloader()))

In [ ]:
with torch.no_grad():
    z = ae_task.encoder(
        input_ids=batch["input_ids_enc"].cuda(), 
        attention_mask=batch["attention_mask_enc"].cuda()
    )

In [ ]:
z = z[:10]

In [ ]:
position_ids = torch.zeros(
    z.shape[0], z.shape[1] + 1, device=z.device, dtype=torch.long
)
bos_emb = ae_task.decoder.backbone.get_input_embeddings()(
    torch.tensor(ae_task.decoder.tokenizer.bos_token_id).cuda()
)
bos_emb = einx.rearrange("d -> b 1 d", bos_emb, b=z.shape[0])
z_with_bos = torch.cat([z, bos_emb], dim=1).half()

In [ ]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        inputs_embeds=z_with_bos,
        #position_ids=position_ids,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
    )

In [ ]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)

In [ ]:
seqs

[' They They had to to to to to to to. to. When had called me and she called her. She said thank you. She said goodbye in housing.',
 ' my my taught me when when. I has. My teacher began to recently. This continues to. This is.',
 ' Tom took a a a a         ',
 ' Martin heard heard a he he he he he he he he he he he he he he had seen a man staring at mask at clown..',
 ' he he he he t he . . . t . t . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . a . . a . . . a . a . a . a . a 1 . a . a C TOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTATERL TOM L T L TAKE TOM L T TATERL T TA MAN TTA TE TOM TOM TOM TOM L T T TTA',
 "Thethe'whenthe the when the to when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when whe

In [ ]:
prefill = ae_task.decoder.backbone(
    inputs_embeds=z.half(),
    use_cache=True,
)
cache = prefill.past_key_values
cache_position = torch.tensor([0])
bos = torch.full(
    (z.shape[0], 1),
    ae_task.decoder.tokenizer.bos_token_id,
    device=z.device,
    dtype=torch.long,
)

In [ ]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        input_ids=bos,
        past_key_values=cache,
        cache_position=cache_position,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
        )

In [ ]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)
seqs

[' they they had to to to to to to to. They And When He There She You She I I I C My advice said said housing was was was. This was was was.',
 ' My grandmother taught me when. I my had. My grandmother had done this projects. I continue to projects. This created.',
 ' Tom took a a a          ',
 ' Martin heard heard a he he he he he he he he he he he he he had a look out of clown face in clown face..',
 ' he he he he          in K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K',
 'TheWhen the the the when the to the the to to to to to.. to. to. to. to, they with ( 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 

In [ ]:
diffusion_task = GaussianDiffusionTask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/a1pq0e97/"
                "last.ckpt",
            ),
            strict=False,
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
diffusion_task = diffusion_task.cuda()

In [ ]:
diffusion_task.setup()

In [ ]:
mauve = diffusion_task.get_mauve_score()

Featurizing q:  88%|████████▊ | 7/8 [00:04<00:00,  1.67it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 214.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 118.44 MiB is free. Including non-PyTorch memory, this process has 44.52 GiB memory in use. Of the allocated memory 39.63 GiB is allocated by PyTorch, and 4.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
mauve

0.11225014485875065

In [ ]:
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
from transformers.models.auto.tokenization_auto import AutoTokenizer


tok = AutoTokenizer.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', model_max_length=512)
model = AutoModelForCausalLM.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', trust_remote_code=True)

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/thesephist/contra-bottleneck-t5-large-wikipedia:
- bottleneck_t5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


In [ ]:
list(model.modules())[3]

ModuleList(
  (0): T5Block(
    (layer): ModuleList(
      (0): T5LayerSelfAttention(
        (SelfAttention): T5Attention(
          (q): Linear(in_features=1024, out_features=1024, bias=False)
          (k): Linear(in_features=1024, out_features=1024, bias=False)
          (v): Linear(in_features=1024, out_features=1024, bias=False)
          (o): Linear(in_features=1024, out_features=1024, bias=False)
          (relative_attention_bias): Embedding(32, 16)
        )
        (layer_norm): T5LayerNorm()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (1): T5LayerFF(
        (DenseReluDense): T5DenseGatedActDense(
          (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
          (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
          (wo): Linear(in_features=2816, out_features=1024, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
          (act): NewGELUActivation()
        )
        (layer_norm): T5LayerNorm()
        (d

In [ ]:
import wandb
from typing import Any, List, Optional, Dict

def get_runs_by_config(
    entity: str,
    project: str,
    config_filters: Dict[str, Any],
    state: Optional[str] = None
) -> List[str]:
    """
    Retrieve run IDs from a W&B project that match specific config values.
    
    Args:
        entity: W&B entity name
        project: W&B project name
        config_filters: Dictionary of config key-value pairs to filter by
                       e.g., {"sweep_id": "dae_sweep", "task.diffusion.model_type": "transformer"}
        state: Optional run state filter ("finished", "running", "crashed", etc.)
    
    Returns:
        List of run IDs that match the criteria
    """
    api = wandb.Api()
    
    # Build the filter dictionary for the API
    filters = {}
    if state:
        filters["state"] = state
    
    # Get all runs first, then filter manually since W&B's config filtering can be unreliable
    runs = api.runs(f"{entity}/{project}", filters=filters)
    
    matching_run_ids = []
    
    for run in runs:
        # Check if all config filters match
        matches_all = True
        for config_key, expected_value in config_filters.items():
            # Navigate nested config using dot notation
            config_value = run.config
            for key_part in config_key.split('.'):
                if isinstance(config_value, dict) and key_part in config_value:
                    config_value = config_value[key_part]
                else:
                    config_value = None
                    break
            
            if config_value != expected_value:
                matches_all = False
                break
        
        if matches_all:
            matching_run_ids.append(run.id)
    
    return matching_run_ids

In [ ]:
def list_run_ids(
    entity: str,
    project: str,
    where: Optional[Dict[str, Any]] = None,
    state: Optional[str] = None,
) -> List[str]:
    api = wandb.Api()
    filters: Dict[str, Any] = {}

    # Convert dot-paths to "config.<dotpath>" for W&B filters
    if where:
        for k, v in where.items():
            filters[f"config.{k}"] = v

    if state:
        filters["state"] = state  # e.g., "finished", "failed", "running", etc.

    runs = api.runs(f"{entity}/{project}", filters=filters)
    return [r.id for r in runs]

In [ ]:
api = wandb.Api()

In [ ]:
filters = {
    "config.sweep_id": "dae_sweep",
}
runs = api.runs(f"guillaume-lajoie/sentence_diffusion", filters=filters)
ids = [run.id for run in runs]

In [ ]:
ae_task.decoder.tokenizer.bos_token_id, ae_task.decoder.tokenizer.eos_token_id, ae_task.decoder.tokenizer.pad_token_id

(50256, 50256, 50257)

In [ ]:
list_run_ids(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    filters={
        "config.sweep_id": "dae_sweep",
    },
    state="finished"
)

[]

In [ ]:
run_ids = get_runs_by_config(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    config_filters={
        "sweep_id": {"$in": ["dae_sweep"]},  # Has diffusion task
    },
    state="finished"
)

In [ ]:
run_ids

[]